In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings("ignore")
import pickle
import os

In [2]:
hormoneReport = pd.read_csv(r"C:\Users\direk\Disease_risk_predictor_-3\SYSTEM\dataset\hormone_report.csv")
hormoneReport = hormoneReport.drop("risk_label", axis=1, errors="ignore")
for col in ["thyroid_disorder_risk", "testosterone_imbalance_risk", "hormonal_imbalance_risk"]:
    if col in hormoneReport.columns:
        hormoneReport = hormoneReport.drop(col, axis=1)
hormoneReport.head()

,tsh,t3,t4,testosterone,estradiol,progesterone,prolactin,lh,fsh,cortisol
0,2.13,92.35,2.51,606.26,-16.47,1.68,9.61,4.84,4.72,6.44
1,1.84,161.66,6.27,229.35,268.30,18.65,9.27,0.87,0.93,19.39
2,2.65,90.88,2.16,537.92,159.07,5.41,11.96,4.71,2.11,10.81
3,3.67,162.46,7.07,565.91,167.70,-4.83,10.30,5.53,3.00,7.37
4,1.63,143.74,5.39,448.22,30.98,-2.98,7.78,3.48,2.78,6.99


In [3]:
def thyroid_disorder(row):
    tsh, t3, t4 = row["tsh"], row["t3"], row["t4"]

    if tsh > 10 or t3 < 50 or t4 < 3:
        return "Critical"
    elif tsh > 6 or t3 < 70 or t4 < 4:
        return "High"
    elif tsh > 4 or t3 < 80 or t4 < 4.5:
        return "Medium"
    else:
        return "Low"


def testosterone_imbalance(v):
    if v < 150:
        return "Critical"
    elif v < 250:
        return "High"
    elif v < 300 or v > 1000:
        return "Medium"
    else:
        return "Low"


def hormonal_imbalance(row):
    score = 0

    if (row[["estradiol", "progesterone", "prolactin", "lh", "fsh", "cortisol"]] < -10).any():
        return "Critical"

    if (row[["estradiol", "progesterone", "prolactin", "lh", "fsh", "cortisol"]] < 0).any():
        return "High"

    if row["prolactin"] > 25:
        score += 1
    if row["cortisol"] > 30:
        score += 1
    if row["estradiol"] > 200:
        score += 1

    ratio = row["lh"] / (row["fsh"] + 1e-5)
    if ratio > 3:
        score += 1

    if score >= 2:
        return "High"
    elif score == 1:
        return "Medium"
    else:
        return "Low"

In [4]:
hormoneReport["thyroid_disorder"] = hormoneReport.apply(thyroid_disorder, axis=1)
hormoneReport["testosterone_imbalance"] = hormoneReport["testosterone"].apply(testosterone_imbalance)
hormoneReport["hormonal_imbalance"] = hormoneReport.apply(hormonal_imbalance, axis=1)
hormoneReport.head()

,tsh,t3,t4,testosterone,estradiol,progesterone,prolactin,lh,fsh,cortisol,thyroid_disorder,testosterone_imbalance,hormonal_imbalance
0,2.13,92.35,2.51,606.26,-16.47,1.68,9.61,4.84,4.72,6.44,Critical,Low,Critical
1,1.84,161.66,6.27,229.35,268.30,18.65,9.27,0.87,0.93,19.39,Low,High,Medium
2,2.65,90.88,2.16,537.92,159.07,5.41,11.96,4.71,2.11,10.81,Critical,Low,Low
3,3.67,162.46,7.07,565.91,167.70,-4.83,10.30,5.53,3.00,7.37,Low,Low,High
4,1.63,143.74,5.39,448.22,30.98,-2.98,7.78,3.48,2.78,6.99,Low,Low,High


In [5]:
LABEL_MAPPING = {
    "Low": "low",
    "Normal": "moderate",
    "Medium": "moderate",
    "High": "high",
    "Critical": "critical"
}

NUM_MAPPING = {
    "low": 0,
    "moderate": 1,
    "high": 2,
    "critical": 3
}

In [6]:
hormoneReport["thyroid_disorder"] = hormoneReport["thyroid_disorder"].map(LABEL_MAPPING)
hormoneReport["testosterone_imbalance"] = hormoneReport["testosterone_imbalance"].map(LABEL_MAPPING)
hormoneReport["hormonal_imbalance"] = hormoneReport["hormonal_imbalance"].map(LABEL_MAPPING)
hormoneReport.head()

,tsh,t3,t4,testosterone,estradiol,progesterone,prolactin,lh,fsh,cortisol,thyroid_disorder,testosterone_imbalance,hormonal_imbalance
0,2.13,92.35,2.51,606.26,-16.47,1.68,9.61,4.84,4.72,6.44,critical,low,critical
1,1.84,161.66,6.27,229.35,268.30,18.65,9.27,0.87,0.93,19.39,low,high,moderate
2,2.65,90.88,2.16,537.92,159.07,5.41,11.96,4.71,2.11,10.81,critical,low,low
3,3.67,162.46,7.07,565.91,167.70,-4.83,10.30,5.53,3.00,7.37,low,low,high
4,1.63,143.74,5.39,448.22,30.98,-2.98,7.78,3.48,2.78,6.99,low,low,high


In [7]:
hormoneReport["thyroid_disorder_num"] = hormoneReport["thyroid_disorder"].map(NUM_MAPPING)
hormoneReport["testosterone_imbalance_num"] = hormoneReport["testosterone_imbalance"].map(NUM_MAPPING)
hormoneReport["hormonal_imbalance_num"] = hormoneReport["hormonal_imbalance"].map(NUM_MAPPING)
hormoneReport.head()

,tsh,t3,t4,testosterone,estradiol,progesterone,prolactin,lh,fsh,cortisol,thyroid_disorder,testosterone_imbalance,hormonal_imbalance,thyroid_disorder_num,testosterone_imbalance_num,hormonal_imbalance_num
0,2.13,92.35,2.51,606.26,-16.47,1.68,9.61,4.84,4.72,6.44,critical,low,critical,3,0,3
1,1.84,161.66,6.27,229.35,268.30,18.65,9.27,0.87,0.93,19.39,low,high,moderate,0,2,1
2,2.65,90.88,2.16,537.92,159.07,5.41,11.96,4.71,2.11,10.81,critical,low,low,3,0,0
3,3.67,162.46,7.07,565.91,167.70,-4.83,10.30,5.53,3.00,7.37,low,low,high,0,0,2
4,1.63,143.74,5.39,448.22,30.98,-2.98,7.78,3.48,2.78,6.99,low,low,high,0,0,2


In [8]:
"""Preparring data for ML prediction"""
feature_cols = ["tsh","t3", "t4", "testosterone", "estradiol", "progesterone", "prolactin", "lh", "fsh", "cortisol"]
X = hormoneReport[feature_cols]
y_thy = hormoneReport["thyroid_disorder_num"]
y_tes = hormoneReport["testosterone_imbalance_num"]
y_hor = hormoneReport["hormonal_imbalance_num"]

In [9]:
"""train test split"""
X_train_thy, X_test_thy, y_train_thy, y_test_thy = train_test_split(X, y_thy, test_size=0.2, random_state=31, stratify=y_thy)
X_train_tes, X_test_tes, y_train_tes, y_test_tes = train_test_split(X, y_tes, test_size=0.2, random_state=31, stratify=y_tes)
X_train_hor, X_test_hor, y_train_hor, y_test_hor = train_test_split(X, y_hor, test_size=0.2, random_state=31, stratify=y_hor)


In [10]:
"""Auto detect classes and print report - works for any number of classes"""
CLASS_NAMES = {0: "low", 1: "moderate", 2: "high", 3: "critical"}

def print_report(y_test, y_pred, model_name):
    # automatically finds which classes exist in test + predictions
    existing_labels = sorted(np.unique(np.concatenate([y_test, y_pred])))
    existing_names = [CLASS_NAMES[i] for i in existing_labels]

    print("=" * 40)
    print(f"{model_name} MODEL ACCURACY")
    print("=" * 40)
    print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
    print()
    print("=" * 40)
    print(f"{model_name} CLASSIFICATION REPORT")
    print("=" * 40)
    print(classification_report(
        y_test, y_pred,
        labels=existing_labels,
        target_names=existing_names
    ))

In [11]:
"""Thyroid disorder model"""
model_thyroid = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,           
    colsample_bytree=0.8,    
    gamma=0.1,               
    use_label_encoder=False,
    eval_metric="mlogloss",
    random_state=31
)
model_thyroid.fit(X_train_thy, y_train_thy)
y_predict_throid = model_thyroid.predict(X_test_thy)
print_report(y_test_thy, y_predict_throid, "THROID DISORDER")

THROID DISORDER MODEL ACCURACY
Accuracy: 98.00%

THROID DISORDER CLASSIFICATION REPORT
              precision    recall  f1-score   support

         low       0.97      1.00      0.99        68
    moderate       1.00      0.94      0.97        16
        high       1.00      0.90      0.95        10
    critical       1.00      1.00      1.00         6

    accuracy                           0.98       100
   macro avg       0.99      0.96      0.98       100
weighted avg       0.98      0.98      0.98       100



In [12]:
"""Testosterone imbalance model"""
model_testosterone = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,           
    colsample_bytree=0.8,    
    gamma=0.1,               
    use_label_encoder=False,
    eval_metric="mlogloss",
    random_state=31
)
model_testosterone.fit(X_train_tes, y_train_tes)
y_predict_testosterone = model_testosterone.predict(X_test_tes)
print_report(y_test_tes, y_predict_testosterone, "TESTOSTERONE IMBALANCE")

TESTOSTERONE IMBALANCE MODEL ACCURACY
Accuracy: 100.00%

TESTOSTERONE IMBALANCE CLASSIFICATION REPORT
              precision    recall  f1-score   support

         low       1.00      1.00      1.00        77
    moderate       1.00      1.00      1.00         7
        high       1.00      1.00      1.00        11
    critical       1.00      1.00      1.00         5

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100



In [13]:
"""Hormonal imbalance model"""
model_hormone = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,           
    colsample_bytree=0.8,    
    gamma=0.1,               
    use_label_encoder=False,
    eval_metric="mlogloss",
    random_state=31
)

model_hormone.fit(X_train_hor, y_train_hor)
y_predict_hormone = model_hormone.predict(X_test_hor)
print_report(y_test_hor, y_predict_hormone, "HORMONAL IMBALANCE")

HORMONAL IMBALANCE MODEL ACCURACY
Accuracy: 98.00%

HORMONAL IMBALANCE CLASSIFICATION REPORT
              precision    recall  f1-score   support

         low       0.96      1.00      0.98        53
    moderate       1.00      0.90      0.95        10
        high       1.00      0.96      0.98        24
    critical       1.00      1.00      1.00        13

    accuracy                           0.98       100
   macro avg       0.99      0.96      0.98       100
weighted avg       0.98      0.98      0.98       100



In [14]:
"""Save both models as pkl"""
save_path = r"C:\Users\direk\Disease_risk_predictor_-3\ml_models\xgboost"
os.makedirs(save_path, exist_ok=True)

with open(os.path.join(save_path, "hormone.pkl"), "wb") as f:
    pickle.dump(model_hormone, f)
    print("hormone.pkl saved successfully!")

with open(os.path.join(save_path, "testosterone.pkl"), "wb") as f:
    pickle.dump(model_testosterone, f)
    print("testosterone.pkl saved sucessfully!")

with open(os.path.join(save_path, "thyroid.pkl"), "wb") as f:
    pickle.dump(model_thyroid, f)
    print("thyroid.pkl saved sucessfully!")


hormone.pkl saved successfully!
testosterone.pkl saved sucessfully!
thyroid.pkl saved sucessfully!
